In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd
import json
import re
from tqdm import tqdm

from src.utilities_screen import *        # use this instead to not substract mean

import warnings
warnings.filterwarnings("ignore")

protein_ctrl = 'NPC'
compound_ctrl = 'NCC'

In [13]:
working_dir = '/Users/behnam/Desktop/My_Codes/screen_analysis/data_GCDH_g3_new_thr/'

In [4]:
# read the layout file and convert it into the well_positions_dict
layout = pd.read_excel(os.path.join(working_dir,"TSA_Layout.xlsx"), index_col=0)
well_positions_dict = layout2dict(layout, working_dir)

In [5]:
# Read the raw data
remove_header(os.path.join(working_dir,'TSA_RawData.csv'))

try:    # type I data with header being removed
    rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData_removed_header.csv'), index_col=0, encoding='ISO-8859-1')
except: # type II and IV data with ; as separator and extra rows to be removed
    try:
        rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData.csv'), encoding='ISO-8859-1', sep=';', index_col = 0)
        rawdata = rawdata.drop(['SampleID'], axis=1)
        pattern = '^[A-Z][0-9]{1,2}$'
        to_keep = [bool(re.findall(pattern, str(i))) for i in rawdata.index]
        rawdata = rawdata.loc[to_keep]
        if len(rawdata.iloc[0,0].split(',')) == 1:       # II
            rawdata = rawdata.apply(pd.to_numeric, errors='coerce')
        else:                                            # IV
            rawdata = rawdata.apply(lambda x: x.str.replace(',', '.').astype(float))
            rawdata.columns = rawdata.columns.str.replace(',', '.')
    except: # type III data with ; as separator and headers
        remove_header(os.path.join(working_dir,'TSA_RawData.csv'), sep=';')
        rawdata = pd.read_csv(os.path.join(working_dir,'TSA_RawData_removed_header.csv'), index_col=0, encoding='ISO-8859-1', sep=';')
        rawdata = rawdata.apply(pd.to_numeric, errors='coerce')

In [6]:
rawdata

,23,23.7,24.4,25.1,25.8,26.5,27.2,27.9,28.6,29.3,...,53.8,54.5,55.2,55.9,56.6,57.3,58,58.7,59.4,60.1
Cy5,,,,,,,,,,,,,,,,,,,,,
A1,7483.716526,7387.165992,7272.932317,7122.107877,7003.199869,6947.376725,6866.417061,6813.865418,6759.043157,6688.079007,...,3368.903176,3317.080899,3277.120142,3240.374985,3173.251987,3069.311960,2969.032075,2889.400710,2817.512874,2731.245340
A2,54398.481144,54413.433154,54409.773013,54325.509906,54148.425616,54265.116244,54443.025729,54649.601441,54693.203706,55244.274578,...,63216.831588,63052.519194,62898.135766,62555.139268,62368.032845,62125.007450,62036.751462,62026.971564,62343.583102,62391.707970
A3,65154.044547,65439.570172,65516.694007,65691.984162,65749.039776,65770.611983,65735.314911,65708.658435,65673.925691,65662.490078,...,69350.501674,68914.079737,68336.054254,67867.183501,67395.911695,66890.633849,66369.842328,65990.478666,65693.629230,65384.156964
A4,66224.155395,66303.770789,66301.026348,66453.828595,66825.994423,66865.292362,67066.403150,67105.610584,67342.005134,67510.326380,...,71547.400991,70747.688071,70114.531544,69688.318074,69186.761593,68719.362882,68316.816551,67870.146219,67496.684036,67262.898168
A5,62823.905439,63291.195011,63522.971128,63704.479531,63761.681550,63659.950918,63502.948159,63354.873483,63246.871365,63188.460834,...,67084.296583,66557.084508,66253.210262,65786.570199,65451.503563,64969.818323,64455.207782,63986.076161,63820.092081,63736.925685
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P20,68108.500815,68010.845770,67938.385412,67879.520343,67902.612055,67928.123442,68060.820474,68250.177071,68374.325499,68698.630200,...,73438.209280,72931.510810,72372.121769,71986.416125,71520.694906,70984.885158,70491.020986,70037.209031,69631.649299,69272.359571
P21,72066.668654,72218.497517,72217.135026,72254.587573,72329.082554,72391.629538,72386.967906,72288.244285,72154.929803,72385.514278,...,73774.398182,73513.893592,73223.550910,72958.749233,72571.350729,72086.101235,71427.268595,70851.374732,70394.427681,69851.579916
P22,64120.282992,64202.269394,64242.642866,64510.976082,64605.281860,64855.326258,65107.924756,65185.579895,65211.398867,65299.942540,...,68950.911999,68591.565311,68229.252130,67948.319292,67449.720579,66806.315070,66203.374171,65607.368760,65033.108519,64667.892640


In [7]:
# Get all proteins, ligands and doses
protein_set = list(well_positions_dict.keys())
compound_set = list(well_positions_dict[protein_set[0]].keys())
try:
    dosage_set = list(well_positions_dict[protein_set[0]][compound_set[0]].keys())
except:
    dosage_set = list(well_positions_dict[protein_set[0]][compound_set[1]].keys())
protein_set, compound_set, dosage_set


(['GCDH'],
 ['A02',
  'A03',
  'A04',
  'A05',
  'A06',
  'A07',
  'A08',
  'A09',
  'A10',
  'A11',
  'A12',
  'A13',
  'A14',
  'A15',
  'A16',
  'A17',
  'A18',
  'A19',
  'A20',
  'A21',
  'A22',
  'needle1-buf',
  'B01',
  'B02',
  'B03',
  'B04',
  'B05',
  'B06',
  'B07',
  'B08',
  'B09',
  'B10',
  'B11',
  'B12',
  'B13',
  'B14',
  'B15',
  'B16',
  'B17',
  'B18',
  'B19',
  'B20',
  'B21',
  'B22',
  'needle1-prot',
  'C02',
  'C03',
  'C04',
  'C05',
  'C06',
  'C07',
  'C08',
  'C09',
  'C10',
  'C11',
  'C12',
  'C13',
  'C14',
  'C15',
  'C16',
  'C17',
  'C18',
  'C19',
  'C20',
  'C21',
  'C22',
  'needle2-buf',
  'D01',
  'D02',
  'D03',
  'D04',
  'D05',
  'D06',
  'D07',
  'D08',
  'D09',
  'D10',
  'D11',
  'D12',
  'D13',
  'D14',
  'D15',
  'D16',
  'D17',
  'D18',
  'D19',
  'D20',
  'D21',
  'D22',
  'needle2-prot',
  'E02',
  'E03',
  'E04',
  'E05',
  'E06',
  'E07',
  'E08',
  'E09',
  'E10',
  'E11',
  'E12',
  'E13',
  'E14',
  'E15',
  'E16',
  'E17',
 

In [8]:
# Get max_replicate
max_replicate = 0
for protein in protein_set:
    for compound in compound_set:
        for dosage in dosage_set:
            # try:
            well_positions_target = well_positions_dict[protein][compound][dosage] if compound != compound_ctrl else well_positions_dict[protein][compound]   
            # except:
            # well_positions_target = []
            if len(well_positions_target) > max_replicate:
                max_replicate = len(well_positions_target)
        
colnames = (['protein', 'compound', 'dosage', 'locations'] +
            [f'tm_{i}' for i in range(1, max_replicate+1)] + ['tm_mean', 'tm_std'] +
            [f'r_squared_{i}' for i in range(1, max_replicate+1)] +
            [f'peak_count_{i}' for i in range(1, max_replicate+1)] + ['peak_count_mean', 'peak_count_std'] +
            [f'fl25_{i}' for i in range(1, max_replicate+1)]
       )

In [10]:
fit_curve = False
df = pd.DataFrame(columns=colnames)
curve = pd.DataFrame()

# Loop over protein
for protein in tqdm(protein_set):
    if protein == protein_ctrl:
        continue
    # Loop over compound
    for compound in compound_set:
        # check for 0 dosage
        if compound == compound_ctrl:
            row, crv = process2(well_positions_dict, rawdata,
                                protein, compound,
                                save_dir = working_dir,
                                fit_curve = fit_curve,
                                max_replicate = max_replicate)
            df.loc[len(df)] = row
            curve = pd.concat([curve, crv], axis=1)
        else:
            # Loop over dosage
            for dosage in dosage_set:
                row, crv = process2(well_positions_dict, rawdata, 
                                    protein, compound, dosage, 
                                    save_dir = working_dir,
                                    fit_curve = fit_curve,
                                    max_replicate = max_replicate)
                df.loc[len(df)] = row
                curve = pd.concat([curve, crv], axis=1)

curve.index = crv['Temperature']
curve = curve.drop('Temperature', axis = 1)

# if not fit_curve:
#     colnames = (['protein', 'compound', 'dosage', 'locations'] +
#                 [f'peak_count_{i}' for i in range(1, max_replicate+1)] + 
#                 ['peak_count_mean', 'peak_count_std'] + 
#                 [f'fl25_{i}' for i in range(1, max_replicate+1)]
#                )
#     df = df[colnames]  

100%|█████████████████████████████████████████████| 1/1 [00:39<00:00, 39.08s/it]


In [11]:
df

,protein,compound,dosage,locations,tm_1,tm_mean,tm_std,r_squared_1,peak_count_1,peak_count_mean,peak_count_std,fl25_1
0,GCDH,A02,dosage,A2,52.263,52.263,0.0,0.0032,2.0,2.0,0.0,54398.4811444219
1,GCDH,A03,dosage,A3,36.4264,36.4264,0.0,0.0034,2.0,2.0,0.0,65154.0445468133
2,GCDH,A04,dosage,A4,36.2635,36.2635,0.0,0.0014,2.0,2.0,0.0,66224.1553953994
3,GCDH,A05,dosage,A5,43.5947,43.5947,0.0,0.0032,2.0,2.0,0.0,62823.9054387422
4,GCDH,A06,dosage,A6,43.5867,43.5867,0.0,0.0126,2.0,2.0,0.0,63360.6445280255
...,...,...,...,...,...,...,...,...,...,...,...,...
353,GCDH,P18,dosage,P18,43.6431,43.6431,0.0,0.0018,2.0,2.0,0.0,75134.312068326
354,GCDH,P19,dosage,P19,39.6748,39.6748,0.0,0.0034,2.0,2.0,0.0,72066.1696813302
355,GCDH,P20,dosage,P20,43.7291,43.7291,0.0,0.0038,2.0,2.0,0.0,68108.5008153143
356,GCDH,P21,dosage,P21,30.0,30.0,0.0,0.0031,2.0,2.0,0.0,72066.6686537029


In [143]:
# df.to_csv("../../data_GCDH/result_table.csv")

In [12]:
df.loc[df['compound'] == 'C04']

,protein,compound,dosage,locations,tm_1,tm_mean,tm_std,r_squared_1,peak_count_1,peak_count_mean,peak_count_std,fl25_1
47,GCDH,C04,dosage,C4,39.7527,39.7527,0.0,0.0041,2.0,2.0,0.0,65677.0717670626
